In [16]:
# ============================================================
# SECTION 1: IMPORT LIBRARIES AND LOAD ORIGINAL DATA
# ============================================================

from google.colab import drive
drive.mount("/content/drive")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import SelectPercentile, f_regression
from sklearn.model_selection import KFold, RandomizedSearchCV, cross_validate

from sklearn.ensemble import GradientBoostingRegressor
from xgboost import XGBRegressor

project_path = Path("/content/drive/MyDrive/ML_Assignment")

# Find the original training CSV
matches = list((project_path / "Data").rglob("train.csv"))

if not matches:
    raise FileNotFoundError(
        "Original train.csv not found. Please put it in the Data folder."
    )

raw_train = pd.read_csv(matches[0])

print("Original dataset:", raw_train.shape)
display(raw_train.head())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Original dataset: (1460, 81)


,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000


In [17]:
# ============================================================
# SECTION 2: DATA PREPARATION
# ============================================================

y = raw_train["SalePrice"]
y_log = np.log1p(y)

X = raw_train.drop(columns=["Id", "SalePrice"]).copy()

# Correct categorical feature types
for col in ["MSSubClass", "MoSold"]:
    X[col] = X[col].astype(str)

# Missing values representing absent property features
none_cols = [
    "PoolQC", "MiscFeature", "Alley", "Fence",
    "FireplaceQu", "GarageType", "GarageFinish",
    "GarageQual", "GarageCond", "BsmtQual",
    "BsmtCond", "BsmtExposure", "BsmtFinType1",
    "BsmtFinType2", "MasVnrType"
]

X[none_cols] = X[none_cols].fillna("None")

# Zero where property component is absent
zero_cols = [
    "GarageCars", "GarageArea",
    "BsmtFinSF1", "BsmtFinSF2", "BsmtUnfSF",
    "TotalBsmtSF", "BsmtFullBath",
    "BsmtHalfBath", "MasVnrArea"
]

X[zero_cols] = X[zero_cols].fillna(0)

# Feature engineering
X["TotalSF"] = (
    X["TotalBsmtSF"] + X["1stFlrSF"] + X["2ndFlrSF"]
)

X["TotalBathrooms"] = (
    X["FullBath"] + 0.5 * X["HalfBath"]
    + X["BsmtFullBath"] + 0.5 * X["BsmtHalfBath"]
)

X["HouseAge"] = X["YrSold"] - X["YearBuilt"]

X["YearsSinceRemodel"] = (
    X["YrSold"] - X["YearRemodAdd"]
)

X["TotalPorchSF"] = X[
    ["OpenPorchSF", "EnclosedPorch", "3SsnPorch",
     "ScreenPorch", "WoodDeckSF"]
].sum(axis=1)

X["HasGarage"] = (X["GarageArea"] > 0).astype(int)
X["HasBasement"] = (X["TotalBsmtSF"] > 0).astype(int)
X["HasFireplace"] = (X["Fireplaces"] > 0).astype(int)
X["HasPool"] = (X["PoolArea"] > 0).astype(int)

print("Prepared dataset:", X.shape)

Prepared dataset: (1460, 88)


In [18]:
# ============================================================
# SECTION 3: PREPROCESSING + FEATURE SELECTION
# ============================================================

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    (
        "num",
        numeric_pipeline,
        make_column_selector(dtype_include=np.number)
    ),
    (
        "cat",
        categorical_pipeline,
        make_column_selector(dtype_include=object)
    )
])

# Feature selection will be fitted separately in each CV fold
selector = SelectPercentile(
    score_func=f_regression,
    percentile=90
)

In [19]:
# ============================================================
# SECTION 4: DEFINE INNER AND OUTER CV
# ============================================================

outer_cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

inner_cv = KFold(
    n_splits=3,
    shuffle=True,
    random_state=43
)

scoring = {
    "RMSE": "neg_root_mean_squared_error",
    "MAE": "neg_mean_absolute_error",
    "R2": "r2"
}

In [20]:
# ============================================================
# SECTION 5: DEFINE CANDIDATE MODELS
# ============================================================

models = {
    "XGBoost": XGBRegressor(
        objective="reg:squarederror",
        tree_method="hist",
        random_state=42,
        n_jobs=1
    ),

    "Gradient Boosting": GradientBoostingRegressor(
        random_state=42
    )
}

search_spaces = {
    "XGBoost": {
        "selector__percentile": [75, 90, 100],
        "model__n_estimators": [200, 300, 500],
        "model__max_depth": [2, 3, 4],
        "model__learning_rate": [0.03, 0.05, 0.1],
        "model__subsample": [0.8, 1.0]
    },

    "Gradient Boosting": {
        "selector__percentile": [75, 90, 100],
        "model__n_estimators": [200, 300, 500],
        "model__max_depth": [2, 3, 4],
        "model__learning_rate": [0.03, 0.05, 0.1],
        "model__subsample": [0.8, 1.0]
    }
}

In [ ]:
# ============================================================
# SECTION 6: RUN NESTED CROSS-VALIDATION
# ============================================================

final_results = []

for name, model in models.items():

    print(f"\nEvaluating {name}...")

    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("selector", SelectPercentile(f_regression)),
        ("model", model)
    ])

    search = RandomizedSearchCV(
        estimator=pipeline,
        param_distributions=search_spaces[name],
        n_iter=8,
        scoring="neg_root_mean_squared_error",
        cv=inner_cv,
        random_state=42,
        n_jobs=1,
        refit=True,
        error_score="raise"
    )

    cv_results = cross_validate(
        search,
        X,
        y_log,
        cv=outer_cv,
        scoring=scoring,
        n_jobs=1,
        error_score="raise"
    )

    final_results.append({
        "Model": name,
        "Mean_RMSE": -cv_results["test_RMSE"].mean(),
        "RMSE_SD": cv_results["test_RMSE"].std(),
        "Mean_MAE": -cv_results["test_MAE"].mean(),
        "Mean_R2": cv_results["test_R2"].mean()
    })

final_results_df = pd.DataFrame(final_results)

final_results_df = final_results_df.sort_values(
    "Mean_RMSE"
)

display(final_results_df.round(4))


Evaluating XGBoost...

Evaluating Gradient Boosting...


,Model,Mean_RMSE,RMSE_SD,Mean_MAE,Mean_R2
1,Gradient Boosting,0.1297,0.0174,0.0853,0.8897
0,XGBoost,0.1311,0.0182,0.0852,0.8872


In [ ]:
# ============================================================
# SECTION 7: SAVE FINAL VALIDATION RESULTS
# ============================================================

output_path = project_path / "Results" / "Outputs"
output_path.mkdir(parents=True, exist_ok=True)

final_results_df.to_csv(
    output_path / "nested_cv_final_results.csv",
    index=False
)

print("Final validation results saved successfully!")

Final validation results saved successfully!


In [ ]:
# ============================================================
# SECTION 8: TRAIN FINAL GRADIENT BOOSTING MODEL
# ============================================================

from sklearn.pipeline import Pipeline
from sklearn.feature_selection import SelectPercentile, f_regression
from sklearn.model_selection import RandomizedSearchCV
from sklearn.ensemble import GradientBoostingRegressor

# Create final pipeline
final_pipeline = Pipeline([
    ("preprocessor", preprocessor),

    ("selector", SelectPercentile(
        score_func=f_regression
    )),

    ("model", GradientBoostingRegressor(
        random_state=42
    ))
])

# Find best hyperparameters using all labelled data
final_search = RandomizedSearchCV(
    estimator=final_pipeline,
    param_distributions=search_spaces["Gradient Boosting"],
    n_iter=8,
    scoring="neg_root_mean_squared_error",
    cv=inner_cv,
    random_state=42,
    n_jobs=1,
    refit=True,
    error_score="raise"
)

final_search.fit(X, y_log)

# Obtain final fitted model
final_model = final_search.best_estimator_

print("Final Gradient Boosting model trained!")
print("\nSelected hyperparameters:")
print(final_search.best_params_)

In [ ]:
# ============================================================
# SECTION 9: SAVE FINAL MODEL
# ============================================================

import joblib

model_path = output_path / "final_gradient_boosting_model.pkl"

joblib.dump(
    final_model,
    model_path
)

print("Final model saved successfully!")
print("Location:", model_path)

Final model saved successfully!
Location: /content/drive/MyDrive/ML_Assignment/Results/Outputs/final_gradient_boosting_model.pkl


In [ ]:
# ============================================================
# SECTION 10: FINAL MODEL FEATURE IMPORTANCE
# ============================================================

import pandas as pd
import numpy as np

# Get fitted preprocessing, selection and model steps
fitted_preprocessor = final_model.named_steps["preprocessor"]
fitted_selector = final_model.named_steps["selector"]
fitted_regressor = final_model.named_steps["model"]

# All feature names after preprocessing and one-hot encoding
all_feature_names = fitted_preprocessor.get_feature_names_out()

# Identify features retained by feature selection
selected_mask = fitted_selector.get_support()
selected_feature_names = all_feature_names[selected_mask]

# Feature importance from final Gradient Boosting model
importance_scores = fitted_regressor.feature_importances_

# Create importance table
feature_importance_df = pd.DataFrame({
    "Feature": selected_feature_names,
    "Importance": importance_scores
})

feature_importance_df = feature_importance_df.sort_values(
    by="Importance",
    ascending=False
).reset_index(drop=True)

print("Features after encoding:", len(all_feature_names))
print("Features selected:", len(selected_feature_names))

display(feature_importance_df.head(20))

Features after encoding: 336
Features selected: 336


,Feature,Importance
0,num__OverallQual,0.363417
1,num__TotalSF,0.337450
2,num__TotalBathrooms,0.030370
3,num__GarageCars,0.017953
4,num__YearsSinceRemodel,0.015565
5,num__LotArea,0.015080
6,num__OverallCond,0.014500
7,num__GarageArea,0.013203
8,num__1stFlrSF,0.011209
9,num__BsmtFinSF1,0.010983


In [ ]:
# ============================================================
# SECTION 11: VISUALIZE TOP 20 FEATURE IMPORTANCES
# ============================================================

import matplotlib.pyplot as plt
import seaborn as sns

top_features = feature_importance_df.head(20)

plt.figure(figsize=(11, 8))

sns.barplot(
    data=top_features,
    x="Importance",
    y="Feature"
)

plt.title("Top 20 Important Features - Final Gradient Boosting")
plt.xlabel("Feature Importance")
plt.ylabel("Feature")

plt.tight_layout()

plt.savefig(
    output_path / "final_feature_importance.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
# ============================================================
# SECTION 12: SAVE FEATURE IMPORTANCE
# ============================================================

feature_importance_df.to_csv(
    output_path / "final_feature_importance.csv",
    index=False
)

print("Feature importance saved successfully!")

In [ ]:
# ============================================================
# SECTION 13: FINAL MODEL PERFORMANCE COMPARISON
# ============================================================

plt.figure(figsize=(8, 5))

sns.barplot(
    data=final_results_df,
    x="Model",
    y="Mean_RMSE"
)

plt.title("Final Nested CV Performance")
plt.xlabel("Model")
plt.ylabel("Mean RMSE (Log Scale)")

plt.tight_layout()
plt.show()

In [21]:
print(final_search.best_params_)

{'selector__percentile': 100, 'model__subsample': 0.8, 'model__n_estimators': 500, 'model__max_depth': 4, 'model__learning_rate': 0.1}


In [22]:
import sys
import pandas as pd
import numpy as np
import sklearn
import xgboost
import hashlib

print("Python:", sys.version.split()[0])
print("Pandas:", pd.__version__)
print("NumPy:", np.__version__)
print("Scikit-learn:", sklearn.__version__)
print("XGBoost:", xgboost.__version__)

with open(matches[0], "rb") as f:
    print("Dataset SHA-256:", hashlib.sha256(f.read()).hexdigest())

Python: 3.13.16
Pandas: 2.2.3
NumPy: 2.1.3
Scikit-learn: 1.6.1
XGBoost: 3.4.1
Dataset SHA-256: 1e18addf81e5e4d347cc17ee6075bbe4a42b7fa26b9e5b063e8f692a5f929d41
